In [1]:
!pip install openai
import os
import sys
import arxiv
from torch import cuda, bfloat16
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from time import time
from langchain.llms import HuggingFacePipeline
from langchain.document_loaders import PyPDFLoader, DirectoryLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.vectorstores import Qdrant
from pathlib import Path
from openai import OpenAI
from IPython.display import Audio, display
from whisperspeech.pipeline import Pipeline

In [2]:
# Create a directory to store the research papers
dirpath = "arxiv_papers"
if not os.path.exists(dirpath):
    os.makedirs(dirpath)

# Search for research papers on ArXiv
search = arxiv.Search(
    query="LLM",
    max_results=10,
    sort_by=arxiv.SortCriterion.LastUpdatedDate,
    sort_order=arxiv.SortOrder.Descending
)

In [3]:
# Download the research papers
for result in search.results():
    while True:
        try:
            result.download_pdf(dirpath=dirpath)
            print(f"-> Paper id {result.get_short_id()} with title '{result.title}' is downloaded.")
            break
        except FileNotFoundError:
            print("File not found")
            break
        except HTTPError:
            print("Forbidden")
            break
        except ConnectionResetError as e:
            print("Connection reset by peer")
            time.sleep(5)

C:\Users\Administrator.DESKTOP-46BVDUV\AppData\Local\Temp\ipykernel_10524\382489390.py:2: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


-> Paper id 2408.07060v1 with title 'Diversity Empowers Intelligence: Integrating Expertise of Software Engineering Agents' is downloaded.
-> Paper id 2408.07055v1 with title 'LongWriter: Unleashing 10,000+ Word Generation from Long Context LLMs' is downloaded.
-> Paper id 2308.04381v3 with title 'Gromov-Wasserstein unsupervised alignment reveals structural correspondences between the color similarity structures of humans and large language models' is downloaded.
-> Paper id 2408.07004v1 with title 'Casper: Prompt Sanitization for Protecting User Privacy in Web-Based Large Language Models' is downloaded.
-> Paper id 2408.07003v1 with title 'Generative AI for automatic topic labelling' is downloaded.
-> Paper id 2408.06993v1 with title 'LLMs can Schedule' is downloaded.
-> Paper id 2402.12261v4 with title 'NEO-BENCH: Evaluating Robustness of Large Language Models with Neologisms' is downloaded.
-> Paper id 2401.02404v4 with title 'Correctness Comparison of ChatGPT-4, Gemini, Claude-3, a

In [4]:

# Load the research papers
loader = DirectoryLoader(dirpath, glob="*.pdf", loader_cls=PyPDFLoader)
papers = loader.load()
print("Total number of pages loaded:", len(papers))

full_text = ''
for paper in papers:
    full_text += paper.page_content.replace('\n', ' ')

print("Total characters in full text:", len(full_text))


Total number of pages loaded: 158
Total characters in full text: 563761


In [5]:
# Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
paper_chunks = text_splitter.create_documents([full_text])
print("Number of chunks created:", len(paper_chunks))

# Clear unused memory
import gc
gc.collect()

Number of chunks created: 1253


14498

In [6]:
from huggingface_hub import login

login()


In [7]:
# Load the Meta-Llama-3-8B-Instruct model
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
device = "cuda"
dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map='auto')




Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
# Create a pipeline for text generation
query_pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    max_length=1024,
    device_map="auto"
)


In [9]:
llm = HuggingFacePipeline(pipeline=query_pipeline)

In [10]:
# Load the sentence transformer model
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}



embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)

    # Alternatively, use a local model
    # local_model_path = "/kaggle/input/sentence-transformers/minilm-l6-v2/all-MiniLM-L6-v2"
    # print(f"Use alternative (local) model: {local_model_path}\n")
    # embeddings = HuggingFaceEmbeddings(model_name=local_model_path, model_kwargs=model_kwargs)

if embeddings is None:
    raise ValueError("Embeddings could not be loaded. Check the exception message above.")


c:\Program Files\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [11]:

# Create a vector database
vectordb = Qdrant.from_documents(
    paper_chunks,
    embeddings,
    path="Qdrant_Persist",
    collection_name="voice_assistant_documents"
)

In [12]:


# Create a retriever
retriever = vectordb.as_retriever()

# Create a QA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True
)

In [13]:
import time
from IPython.display import display, Markdown

def test_rag(qa, query, max_chars=200):
    time_start = time.time()
    response = qa.run(query)
    time_end = time.time()
    total_time = f"{round(time_end - time_start, 3)} sec."

    # Truncate response if it exceeds max_chars
    if len(response) > max_chars:
        response = response[:max_chars] + "..."

    full_response = f"Question: {query}\nAnswer: {response}\nTotal time: {total_time}"
    display(Markdown(full_response))
    return response


In [14]:
# Define a function to colorize text
def colorize_text(text):
    for word, color in zip(["Reasoning", "Question", "Answer", "Total time"], ["blue", "red", "green", "magenta"]):
        text = text.replace(f"{word}:", f"\n\n**<font color='{color}'>{word}:</font>**")
    return text


In [15]:
# Load the Whisperspeech pipeline
pipe = Pipeline(s2a_ref='collabora/whisperspeech:s2a-q4-tiny-en+pl.model')

# Define a query and test the QA chain
# Example usage


# Load model with reduced precision

# Replace MockQASystem with your actual QA system instance

query = "how is llm trained"
aud = test_rag(qa, query )


c:\Program Files\Python312\Lib\site-packages\whisperspeech\t2s_up_wds_mlang_enclm.py:337: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  spec = torch.load(local_filename, map



> Entering new RetrievalQA chain...


c:\Program Files\Python312\Lib\site-packages\transformers\models\llama\modeling_llama.py:660: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



> Finished chain.


Question: how is llm trained
Answer:  The text does not explicitly state how LLM (Large Language Model) is trained. However, it mentions that training a base LLM from scratch typically demands a substantial amount of data and significant...
Total time: 103.814 sec.

In [16]:
print(aud)

 The text does not explicitly state how LLM (Large Language Model) is trained. However, it mentions that training a base LLM from scratch typically demands a substantial amount of data and significant...


In [17]:
# Generate audio using Whisperspeech
pipe.generate_to_notebook(f"{aud}")

c:\Program Files\Python312\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
